# Implementation of Convoluntional Neural Network for Binary Classification of Chest X-Rays (PneumoniaMNIST)

Convolutional neural networks (CNNs) are a common solution for classification in medical imaging.

The **PneumoniaMNIST** dataset from the [MedMNIST](https://medmnist.com/) collection contains 28x28 grayscale chest X-ray images split into training, validation and test sets. Given the set of chest X-ray images belonging to two classes, the CNN will learn the characteristics in the image that let us distinguish the classes: **normal** lungs versus lungs showing **pneumonia**.

## Set up

Start by installing the `medmnist` package and downloading the dataset.# New Section

In [ ]:
#pip install medmnist

## Load Data and Prepare for modeling with PyTorch

In [ ]:
import medmnist
from medmnist import PneumoniaMNIST
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [ ]:
# Define any transformations (e.g., convert to PyTorch tensors)
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])


In [ ]:
# 1. Download and load the datasets
train_dataset = PneumoniaMNIST(split='train', transform=data_transform, download=True)
val_dataset = PneumoniaMNIST(split='val', transform=data_transform, download=True)
test_dataset = PneumoniaMNIST(split='test', transform=data_transform, download=True)


In [ ]:
# 2. Wrap them in PyTorch DataLoaders for your model
batch_size = 128
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)


## Define the CNN class using PyTorch

In [ ]:
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Define the Model Architecture ------------------------------------------------
# global options
num_classes = 1  # binary classification, single logit output

# network architecture options
conv1_filters       = 16
conv1_filter_size   = 3  # can consider optimizing kernel size e.g. 5x5 broader
conv2_filters       = 32
conv2_filter_size   = 3
maxpool_width       = 2
dense_hidden_units  = 64
dropout_rate        = 0.3

# construct the model ----------------------------------------------------------
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, conv1_filters, kernel_size = conv1_filter_size, padding = 'same'),
            nn.BatchNorm2d(conv1_filters),  # normalizes the outputs of each conv layer
            nn.ReLU(),
            nn.MaxPool2d(maxpool_width),

            nn.Conv2d(conv1_filters, conv2_filters, kernel_size = conv2_filter_size, padding = 'same'),
            nn.BatchNorm2d(conv2_filters),  # normalizes the outputs of each conv layer
            nn.ReLU(),
            nn.MaxPool2d(maxpool_width),
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(conv2_filters * 7 * 7, dense_hidden_units),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(dense_hidden_units, num_classes)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

model = CNN().to(device)
print(model)


Create loss function and optimizer. Adam implements per-parameter adaptive learning rates. https://arxiv.org/abs/1412.6980

In [ ]:
import torch.optim as optim

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

## Training

The CNN is trained over multiple epochs on the training set, updating its weights to minimize classification error. Evaluation on the validation set after each epoch is included to monitor generalization and catch overfitting.

In [ ]:

num_epochs = 10

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(num_epochs):
    # -- train --#
    model.train()
    train_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device).float()

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += images.size(0)

    train_loss /= total
    train_acc = correct / total

    # -- validate -- #
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device).float()

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += images.size(0)

    val_loss /= total
    val_acc = correct / total

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    # show progress in each epoch
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

## Visualize Training Accuracy and Loss

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(history['train_loss'], label='train')
plt.plot(history['val_loss'], label='validation')
plt.title('Loss over epochs')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.show()

plt.figure()
plt.plot(history['train_acc'], label='train')
plt.plot(history['val_acc'], label='validation')
plt.title('Accuracy over epochs')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.legend()
plt.show()

## Final Evaluation

The trained model is evaluated below with a held-out test set, which was never used during training or validation.

This gives an unbiased estimate of the model performance on unseen data. Precision, recall, and F1-score are reported per class. A confusion matrix details classification of each case, and ROC-AUC is used as a measure of the model's ability to distinguish pneumonia from normal cases.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

model.eval()
all_labels, all_preds, all_probs = [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device).float()

        outputs = model(images)
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()

        all_labels.extend(labels.cpu().numpy().flatten())
        all_preds.extend(preds.cpu().numpy().flatten())
        all_probs.extend(probs.cpu().numpy().flatten())

print(classification_report(all_labels, all_preds, target_names=['Normal', 'Pneumonia']))
print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))
print(f"ROC-AUC: {roc_auc_score(all_labels, all_probs):.4f}")